In [ ]:
import pandas as pd
import openai
import os
from dotenv import load_dotenv

# .env 파일에 정의된 환경 변수를 로드합니다.
load_dotenv()

# 로드된 환경 변수에서 API 키를 가져옵니다.
upstage_api_key = os.getenv("UPSTAGE_API_KEY")

In [ ]:
# Upstage API와 통신할 클라이언트를 초기화합니다.
# 제공해주신 예시 코드를 기반으로 설정합니다.
client = openai.OpenAI(
    base_url="https://api.upstage.ai/v1",
    api_key=upstage_api_key,
)

In [ ]:
response = client.chat.completions.create(
    model="solar-pro2",
    messages=[
        {
            "role": "user",
            "content": "동원 참치가 뭔지 알아?"
        }
    ]
)
    # 응답을 출력합니다.
print(response.choices[0].message.content)

1. 브랜드
2. 내용물 종류(살코기, 고추참치, 야채참치)
3. 용량
4. 가격

## 양반, 동원, 덴마크, 그릴리, 비비드,  만 뽑고 나머지 drop (GNC)

In [ ]:
import pandas as pd
import numpy as np

df= pd.read_csv('data/학습용.csv', encoding='utf-8')

In [ ]:
df.info()

In [ ]:
# df의 상품명 컬럼에 양반, 동원, 덴마크, 그릴리, 비비드 중에 하나라도 들어간 행은 살려두고 나머지 열은 drop하고 인덱스 정리
df = df[df['상품명'].str.contains('양반|동원|덴마크|그릴리|비비드', na=False)]
df.reset_index(drop=True, inplace=True)
df

In [ ]:
# 1차 카테고리의 모든 종류 확인
df['1차 카테고리'].unique()

In [ ]:
df = df[~df['1차 카테고리'].isin(['반려동식물', '주방용품', '생활/유아동', '과일/채소/쌀', '건강식품'])]
df.reset_index(drop=True, inplace=True)


In [ ]:
# df에서 1차 카테고리와 2차 카테고리와 3차 카테고리 컬럼과 페이지 컬럼 제거
df = df.drop(columns=['1차 카테고리', '2차 카테고리', '3차 카테고리', '페이지'])
df

In [ ]:
df.to_csv('data/1차_컨조인트_정리.csv', index=False, encoding='utf-8-sig')

In [ ]:
df1 = pd.read_csv('data/1차_컨조인트_정리.csv', encoding='utf-8-sig')

In [ ]:
df1.info()

In [ ]:
df1.tail(10)

In [ ]:
df1.drop(columns=['평점', '리뷰 수'], inplace=True)

In [ ]:
df1

In [ ]:
df1.to_csv('data/학습용_수정.csv', index=False, encoding='utf-8-sig')

- 참치캔 구매 상황 컨조인트 분석 설계
- 속성 및 속성 수준 <br>
속성 1: 참치 종류

수준: 살코기 참치, 고추 참치, 야채 참치

속성 2: 캔 용량

수준: 85g, 150g, 250g

속성 3: 묶음 수량

수준: 3개입, 10개입, 20개입

속성 4: 가격

수준: 10,000원, 25,000원, 45,000원

In [ ]:
# 3차 카테고리.1 컬럼을 3차 카테고리로 변경
df1.rename(columns={'3차 카테고리.1': '3차 카테고리'}, inplace=True)
df1['3차 카테고리'].unique()

In [ ]:
# '3차 카테고리' -> '식품유형' 매핑 규칙
food_type_map = {
    '참치/연어통조림': '통조림',
    '햄/닭가슴살': '가공육',
    '국/탕/찌개': '즉석조리식품',
    '죽/스프': '즉석조리식품',
    '골뱅이/수산통조림': '통조림',
    '기타양념': '양념',
    '고등어/꽁치통조림': '통조림',
    '과일/옥수수/팥': '통조림',
    '아이반찬': '반찬',
    '동원참치액': '소스',
    '떡볶이/떡류': '떡류',
    '소금': '양념',
    '반찬/기타통조림': '반찬',
    '고춧가루/액젓': '양념',
    '분말/액상조미료': '조미료',
    '간장/고추장/된장/쌈장': '장류',
    '파스타/생면/건면/기타 면': '면류',
    '올리고당/물엿/조청/꿀': '당류',
    '부침가루/튀김가루/빵가루': '가루류',
    '생지/냉동반죽': '냉동식품',
    '짜장': '소스',
    '메인요리': '즉석조리식품',
    '고기/찌개양념': '양념',
    '밑반찬': '반찬',
    '봉지라면': '라면',
    '한식': '즉석조리식품',
    '카레': '소스',
    '젓갈': '젓갈',
    '찹쌀가루/전분/기타가루': '가루류'
}

# '3차 카테고리' -> '용도' 매핑 규칙
usage_map = {
    '참치/연어통조림': '반찬/주재료',
    '햄/닭가슴살': '반찬/주재료',
    '국/탕/찌개': '간편식사',
    '죽/스프': '간편식사',
    '골뱅이/수산통조림': '안주/반찬',
    '기타양념': '조미료',
    '고등어/꽁치통조림': '반찬',
    '과일/옥수수/팥': '간식/요리재료',
    '아이반찬': '반찬',
    '동원참치액': '조미료',
    '떡볶이/떡류': '간식',
    '소금': '조미료',
    '반찬/기타통조림': '반찬',
    '고춧가루/액젓': '조미료',
    '분말/액상조미료': '조미료',
    '간장/고추장/된장/쌈장': '조미료',
    '파스타/생면/건면/기타 면': '주식',
    '올리고당/물엿/조청/꿀': '요리재료',
    '부침가루/튀김가루/빵가루': '요리재료',
    '생지/냉동반죽': '요리재료',
    '짜장': '요리재료',
    '메인요리': '메인요리',
    '고기/찌개양념': '조미료',
    '밑반찬': '반찬',
    '봉지라면': '간편식사',
    '한식': '메인요리',
    '카레': '요리재료',
    '젓갈': '반찬',
    '찹쌀가루/전분/기타가루': '요리재료'
}

In [ ]:
mapped_food_types = df1['3차 카테고리'].map(food_type_map)
mapped_usages = df1['3차 카테고리'].map(usage_map)


df1['식품유형'] = df1['식품유형'].fillna(mapped_food_types)
df1['용도'] = df1['용도'].fillna(mapped_usages)
print("'식품유형' 및 '용도'의 빈 값을 모두 채움")


print("\n--- 최종 결측치 확인 ---")
print(df1[['식품유형', '용도']].isnull().sum())

print("\n--- 데이터 변경 결과 (상위 10개) ---")
print(df1[['상품명', '3차 카테고리', '식품유형', '용도']].head(10))

output_file_path = 'data/학습용_카테고리_채우기_완료.csv'
df1.to_csv(output_file_path, index=False, encoding='utf-8-sig')
print(f"\n '{output_file_path}' 파일로 저장")

In [ ]:
import pandas as pd
import numpy as np
import re
from math import prod

print("데이터 처리 시작 (최종 수정)...")

try:
    # 1. 원본 데이터 불러오기
    df = pd.read_csv('학습용_수정.csv')
    print("✅ 원본 파일을 성공적으로 불러왔습니다.")

    # --- 최종 계산 함수 ---
    def get_final_calculations(row):
        name = str(row['상품명'])
        weight_str = str(row['중량 수치.1'])
        unit_str = str(row['중량 단위'])
        price = row['가격']
        
        # 기본값 및 초기화
        weight, quantity, total_weight, price_per_10g = np.nan, 1, np.nan, np.nan

        # 규칙 0: 복합 상품은 계산하지 않음
        if '+' in name or (isinstance(weight_str, str) and ',' in weight_str):
            return pd.Series([np.nan, np.nan, np.nan, np.nan])

        # 규칙 1: 무게 확정 (소수점 오류 수정)
        weight = pd.to_numeric(weight_str, errors='coerce')

        # 규칙 2: 묶음수량 확정 (괄호 오류 수정)
        # 괄호와 그 안의 내용을 모두 제거하여 계산에 영향을 주지 않도록 함
        name_no_parentheses = re.sub(r'\(.*\)', '', name)
        
        total_match = re.search(r'총\s*(\d+)', name_no_parentheses)
        if total_match:
            quantity = int(total_match.group(1))
        else:
            name_no_weight = re.sub(r'\b\d+(\.\d+)?\s*[gGkK][gG]\b', '', name_no_parentheses)
            q_units = r'(?:개|캔|병|봉|팩|입|박스)'
            q_matches = re.findall(r'(\d+)\s*' + q_units, name_no_weight)
            
            if q_matches:
                quantity = prod(int(n) for n in q_matches)
            else:
                x_matches = re.findall(r'[\sx\*]\s*(\d+)', name_no_weight)
                if x_matches:
                    quantity = prod(int(n) for n in x_matches)
                else:
                    quantity = 1
        
        # 규칙 3: 최종 계산
        if pd.notna(weight) and pd.notna(quantity):
            conversion_factor = 1000 if 'kg' in unit_str.lower() else 1
            total_weight = weight * quantity * conversion_factor
            if total_weight > 0:
                price_per_10g = round((price / total_weight) * 10, 2)

        return pd.Series([weight, quantity, total_weight, price_per_10g])

    # --- 모든 계산 다시 수행 ---
    df[['중량 수치', '묶음수량', '총중량', '10g당 가격']] = df.apply(get_final_calculations, axis=1, result_type='expand')
    
    # --- 최종 정리 ---
    food_type_map={'참치/연어통조림':'통조림','햄/닭가슴살':'가공육','국/탕/찌개':'즉석조리식품','죽/스프':'즉석조리식품','골뱅이/수산통조림':'통조림','기타양념':'양념','고등어/꽁치통조림':'통조림','과일/옥수수/팥':'통조림','아이반찬':'반찬','동원참치액':'소스','떡볶이/떡류':'떡류','소금':'양념','반찬/기타통조림':'반찬','고춧가루/액젓':'양념','분말/액상조미료':'조미료','간장/고추장/된장/쌈장':'장류','파스타/생면/건면/기타 면':'면류','올리고당/물엿/조청/꿀':'당류','부침가루/튀김가루/빵가루':'가루류','생지/냉동반죽':'냉동식품','짜장':'소스','메인요리':'즉석조리식품','고기/찌개양념':'양념','밑반찬':'반찬','봉지라면':'라면','한식':'즉석조리식품','카레':'소스','젓갈':'젓갈','찹쌀가루/전분/기타가루':'가루류'}
    usage_map={'참치/연어통조림':'반찬/주재료','햄/닭가슴살':'반찬/주재료','국/탕/찌개':'간편식사','죽/스프':'간편식사','골뱅이/수산통조림':'안주/반찬','기타양념':'조미료','고등어/꽁치통조림':'반찬','과일/옥수수/팥':'간식/요리재료','아이반찬':'반찬','동원참치액':'조미료','떡볶이/떡류':'간식','소금':'조미료','반찬/기타통조림':'반찬','고춧가루/액젓':'조미료','분말/액상조미료':'조미료','간장/고추장/된장/쌈장':'조미료','파스타/생면/건면/기타 면':'주식','올리고당/물엿/조청/꿀':'요리재료','부침가루/튀김가루/빵가루':'요리재료','생지/냉동반죽':'요리재료','짜장':'요리재료','메인요리':'메인요리','고기/찌개양념':'조미료','밑반찬':'반찬','봉지라면':'간편식사','한식':'메인요리','카레':'요리재료','젓갈':'반찬','찹쌀가루/전분/기타가루':'요리재료'}
    df['식품유형'] = df['식품유형'].fillna(df['3차 카테고리.1'].map(food_type_map))
    df['용도'] = df['용도'].fillna(df['3차 카테고리.1'].map(usage_map))
    df.rename(columns={'3차 카테고리.1': '3차 카테고리'}, inplace=True)
    
    final_columns = ['상품명', '3차 카테고리', '식품유형', '용도', '가격', '평점', '리뷰 수', '중량 수치', '묶음수량', '총중량', '10g당 가격']
    final_df = df[final_columns].copy()
    
    output_path = '진짜_최종_완성본2.csv'
    final_df.to_csv(output_path, index=False, encoding='utf-8-sig')

    print(f"\n🎉 모든 작업 완료! '{output_path}' 이름으로 파일이 저장되었습니다.")

except FileNotFoundError:
    print("\n❌ 오류: '학습용_수정.csv' 파일을 찾을 수 없습니다.")
except Exception as e:
    print(f"\n❌ 예상치 못한 오류가 발생했습니다: {e}")

In [ ]:
import pandas as pd
import numpy as np
import re
from math import prod

print("데이터 처리 시작 (최종 단순화 버전)...")

try:
    # 1. 원본 데이터 불러오기
    df = pd.read_csv('학습용_수정.csv')
    print("✅ 원본 파일을 성공적으로 불러왔습니다.")

    # --- 최종 계산 함수 ---
    def get_final_calculations(row):
        name = str(row['상품명'])
        weight_str = str(row['중량 수치.1'])
        unit_str = str(row['중량 단위'])
        price = row['가격']
        
        # 기본값 및 초기화
        weight, quantity, total_weight, price_per_10g = np.nan, 1, np.nan, np.nan

        # 규칙 0: 복합 상품은 계산하지 않음
        if '+' in name or (isinstance(weight_str, str) and ',' in weight_str):
            return pd.Series([np.nan, np.nan, np.nan, np.nan])

        # 규칙 1: 무게 확정
        weight = pd.to_numeric(weight_str, errors='coerce')

        # 규칙 2: 묶음수량 확정
        # 2-1: '총' 키워드가 있으면 최우선 적용
        total_match = re.search(r'총\s*(\d+)', name)
        if total_match:
            quantity = int(total_match.group(1))
        else:
            # 2-2: 무게와 괄호 정보를 완전히 제거
            name_no_parentheses = re.sub(r'\(.*\)', '', name)
            name_no_weight = re.sub(r'\b\d+(\.\d+)?\s*[gGkK][gG]\b', '', name_no_parentheses)
            
            # 2-3: '개, 캔, 박스' 등 명확한 수량 단위가 있는 숫자만 찾아 곱함
            quantity_units = r'(?:개|캔|병|봉|팩|입|박스)'
            quantity_matches = re.findall(r'(\d+)\s*' + quantity_units, name_no_weight)
            
            if quantity_matches:
                quantity = prod(int(n) for n in quantity_matches)
            else:
                # 2-4: 'x' 뒤에 있는 숫자 찾기 (수량 단위 없을 때)
                x_matches = re.findall(r'[\sx\*]\s*(\d+)', name_no_weight)
                if x_matches:
                     quantity = prod(int(n) for n in x_matches)
                else:
                    # 2-5: 위 규칙에 없으면 무조건 1개로 처리
                    quantity = 1
        
        # 규칙 3: 최종 계산
        if pd.notna(weight) and pd.notna(quantity):
            conversion_factor = 1000 if 'kg' in unit_str.lower() else 1
            total_weight = weight * quantity * conversion_factor
            if total_weight > 0:
                price_per_10g = round((price / total_weight) * 10, 2)

        return pd.Series([weight, quantity, total_weight, price_per_10g])

    # --- 모든 계산 다시 수행 ---
    calculated_df = df.apply(get_final_calculations, axis=1, result_type='expand')
    df[['중량 수치', '묶음수량', '총중량', '10g당 가격']] = calculated_df
    
    # --- 최종 정리 ---
    food_type_map={'참치/연어통조림':'통조림','햄/닭가슴살':'가공육','국/탕/찌개':'즉석조리식품','죽/스프':'즉석조리식품','골뱅이/수산통조림':'통조림','기타양념':'양념','고등어/꽁치통조림':'통조림','과일/옥수수/팥':'통조림','아이반찬':'반찬','동원참치액':'소스','떡볶이/떡류':'떡류','소금':'양념','반찬/기타통조림':'반찬','고춧가루/액젓':'양념','분말/액상조미료':'조미료','간장/고추장/된장/쌈장':'장류','파스타/생면/건면/기타 면':'면류','올리고당/물엿/조청/꿀':'당류','부침가루/튀김가루/빵가루':'가루류','생지/냉동반죽':'냉동식품','짜장':'소스','메인요리':'즉석조리식품','고기/찌개양념':'양념','밑반찬':'반찬','봉지라면':'라면','한식':'즉석조리식품','카레':'소스','젓갈':'젓갈','찹쌀가루/전분/기타가루':'가루류'}
    usage_map={'참치/연어통조림':'반찬/주재료','햄/닭가슴살':'반찬/주재료','국/탕/찌개':'간편식사','죽/스프':'간편식사','골뱅이/수산통조림':'안주/반찬','기타양념':'조미료','고등어/꽁치통조림':'반찬','과일/옥수수/팥':'간식/요리재료','아이반찬':'반찬','동원참치액':'조미료','떡볶이/떡류':'간식','소금':'조미료','반찬/기타통조림':'반찬','고춧가루/액젓':'조미료','분말/액상조미료':'조미료','간장/고추장/된장/쌈장':'조미료','파스타/생면/건면/기타 면':'주식','올리고당/물엿/조청/꿀':'요리재료','부침가루/튀김가루/빵가루':'요리재료','생지/냉동반죽':'요리재료','짜장':'요리재료','메인요리':'메인요리','고기/찌개양념':'조미료','밑반찬':'반찬','봉지라면':'간편식사','한식':'메인요리','카레':'요리재료','젓갈':'반찬','찹쌀가루/전분/기타가루':'요리재료'}
    df['식품유형'] = df['식품유형'].fillna(df['3차 카테고리.1'].map(food_type_map))
    df['용도'] = df['용도'].fillna(df['3차 카테고리.1'].map(usage_map))
    df.rename(columns={'3차 카테고리.1': '3차 카테고리'}, inplace=True)
    
    final_columns = ['상품명', '3차 카테고리', '식품유형', '용도', '가격', '평점', '리뷰 수', '중량 수치', '묶음수량', '총중량', '10g당 가격']
    final_df = df[final_columns].copy()
    
    output_path = '최종_완성본4.csv'
    final_df.to_csv(output_path, index=False, encoding='utf-8-sig')

    print(f"\n🎉 모든 작업 완료! '{output_path}' 이름으로 파일이 저장되었습니다.")

except FileNotFoundError:
    print("\n❌ 오류: '학습용_수정.csv' 파일을 찾을 수 없습니다.")
except Exception as e:
    print(f"\n❌ 예상치 못한 오류가 발생했습니다: {e}")